In [17]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CreditCardFraudDetection") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR") 
print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [18]:
from pyspark.sql.functions import lit

# Loading Training Data (80%)
df_train = spark.read.csv("../data/fraudTrain.csv", header=True, inferSchema=True)
df_train = df_train.withColumn("split", lit("train"))

print("Train Data Loaded! Rows:", df_train.count())


Train Data Loaded! Rows: 1296675


In [25]:
from pyspark.sql.functions import col

# Load fraudTest.csv (this is ~20% of total data)
df_test_full = spark.read.csv("../data/fraudTest.csv", header=True, inferSchema=True)

total_test_rows = df_test_full.count()
stream_start = int(total_test_rows * 0.5)  # split 50/50

print("Total test file rows:", total_test_rows)
print("Streaming starts at row:", stream_start)

# Split using _c0 (existing row index column)
df_test   = df_test_full.filter(col("_c0") < stream_start)
df_stream = df_test_full.filter(col("_c0") >= stream_start)

# Add split labels
df_test   = df_test.withColumn("split", lit("test"))
df_stream = df_stream.withColumn("split", lit("stream"))

print("Test rows:", df_test.count())
print("Stream rows:", df_stream.count())

Total test file rows: 555719
Streaming starts at row: 277859
Test rows: 277859
Stream rows: 277860


In [8]:
df_test_full.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- trans_date_trans_time: timestamp (nullable = true)
 |-- cc_num: long (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: integer (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- is_fraud: integer (nullable = true)



In [19]:
# Save streaming data for streaming_demo.ipynb
df_stream.coalesce(1).write.csv("../data/streaming_data", header=True, mode="overwrite")

print("Streaming data saved to ../data/streaming_data/")

Streaming data saved to ../data/streaming_data/


In [20]:
# Final batch ingestion dataframe (train + test only)
df = df_train.union(df_test)

print("Datasets combined successfully!")
print("Total batch rows:", df.count())

Datasets combined successfully!
Total batch rows: 1574534


In [11]:
# Print Schema
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- trans_date_trans_time: timestamp (nullable = true)
 |-- cc_num: long (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: integer (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- split: string (nullable = false)



In [21]:
# Show first 5 rows
df.show(5)

+---+---------------------+----------------+--------------------+-------------+------+---------+-------+------+--------------------+--------------+-----+-----+-------+---------+--------+--------------------+----------+--------------------+----------+------------------+-----------+--------+-----+
|_c0|trans_date_trans_time|          cc_num|            merchant|     category|   amt|    first|   last|gender|              street|          city|state|  zip|    lat|     long|city_pop|                 job|       dob|           trans_num| unix_time|         merch_lat| merch_long|is_fraud|split|
+---+---------------------+----------------+--------------------+-------------+------+---------+-------+------+--------------------+--------------+-----+-----+-------+---------+--------+--------------------+----------+--------------------+----------+------------------+-----------+--------+-----+
|  0|  2019-01-01 00:00:18|2703186189652095|fraud_Rippin, Kub...|     misc_net|  4.97| Jennifer|  Banks|     

In [22]:
from pyspark.sql.functions import col, sum as spark_sum

# Check missing values in each column
missing_values = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df.columns
])
missing_values.show()

+---+---------------------+------+--------+--------+---+-----+----+------+------+----+-----+---+---+----+--------+---+---+---------+---------+---------+----------+--------+-----+
|_c0|trans_date_trans_time|cc_num|merchant|category|amt|first|last|gender|street|city|state|zip|lat|long|city_pop|job|dob|trans_num|unix_time|merch_lat|merch_long|is_fraud|split|
+---+---------------------+------+--------+--------+---+-----+----+------+------+----+-----+---+---+----+--------+---+---+---------+---------+---------+----------+--------+-----+
|  0|                    0|     0|       0|       0|  0|    0|   0|     0|     0|   0|    0|  0|  0|   0|       0|  0|  0|        0|        0|        0|         0|       0|    0|
+---+---------------------+------+--------+--------+---+-----+----+------+------+----+-----+---+---+----+--------+---+---+---------+---------+---------+----------+--------+-----+



In [23]:
# Basic Statistics
df.describe().show()

+-------+------------------+--------------------+-------------------+-------------+-----------------+-------+-------+-------+--------------------+-------+-------+------------------+-----------------+------------------+------------------+------------------+--------------------+--------------------+-----------------+------------------+--------------------+-------+
|summary|               _c0|              cc_num|           merchant|     category|              amt|  first|   last| gender|              street|   city|  state|               zip|              lat|              long|          city_pop|               job|           trans_num|           unix_time|        merch_lat|        merch_long|            is_fraud|  split|
+-------+------------------+--------------------+-------------------+-------------+-----------------+-------+-------+-------+--------------------+-------+-------+------------------+-----------------+------------------+------------------+------------------+--------------

In [24]:
# Row and Column Count
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

Total Rows: 1574534
Total Columns: 24
